**The purpose of this notebook:** apply the clean function and attend the specific needs of the dataset to The final cleaned dataset to be used for further analysis and modeling.


In [1]:
import pandas as pd
import yaml
import openpyxl

---

In [2]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config").is_dir())
with open(ROOT / "config" / "config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg = {k: str(ROOT / v) for k, v in cfg.items()}  # relative paths resolve against the project root

In [4]:
P_A1_25 = pd.read_csv(cfg["P_A1_25_function_tested_csv_path"], encoding="utf-8-sig")

In [5]:
# How many non-zero values each column has and what percentage of all rows that represents.
non_zero_counts = (P_A1_25 != 0.0).sum()
non_zero_pct = non_zero_counts / len(P_A1_25) * 100

pd.DataFrame({
    "non_zero_count": non_zero_counts,
    "non_zero_pct": non_zero_pct.round(2)
})

,non_zero_count,non_zero_pct
\n \n Fecha,7524,100.00
\n \n Fórmula,7524,100.00
\n \n Volumen,7522,99.97
"Cemento\n 1\n CEM I 42,5R",7470,99.28
Cemento\n 7\n CEM IV/A(V),14,0.19
Arido\n 1\n ARENA 0-4,7487,99.51
Arido\n 2\n ARIDO 8-14,7244,96.28
Arido\n 3\n ARIDO 12-20,5565,73.96
Agua\n 1\n AGUA,7362,97.85
Agua\n 2\n AGUA RECICLA,0,0.00


In [6]:
P_A1_25.shape  # rows and columns count

(7524, 13)

In [7]:
P_A1_25.head(5)

,\n \n Fecha,\n \n Fórmula,\n \n Volumen,"Cemento\n 1\n CEM I 42,5R",Cemento\n 7\n CEM IV/A(V),Arido\n 1\n ARENA 0-4,Arido\n 2\n ARIDO 8-14,Arido\n 3\n ARIDO 12-20,Agua\n 1\n AGUA,Agua\n 2\n AGUA RECICLA,Aditivo\n 1\n COMPLAST MR-,Aditivo\n 3\n DYNAMON SX-3,Aditivo\n 4\n AURAMIX 269
0,2025-12-30,HA-25/B/20/IIa/1,6.0,1748.0,0.0,7165.0,1725.0,3794.0,733.0,0.0,0.0,7.4,10.8
1,2025-12-30,HA-25/B/20/XC1/1,5.0,1385.0,0.0,5843.0,1456.0,3136.0,606.0,0.0,0.0,6.4,9.0
2,2025-12-30,HA-25/B/20/XC1/1,4.5,1246.0,0.0,5363.0,1298.0,2866.0,518.0,0.0,0.0,5.6,8.1
3,2025-12-30,HM-20/B/20/I/1,4.0,807.0,0.0,4910.0,1372.0,2157.0,392.0,0.0,0.0,0.0,8.0
4,2025-12-30,HA-30/F/14/XC1/1,3.5,1055.0,0.0,4258.0,2974.0,0.0,464.0,0.0,0.0,4.8,6.8


In [8]:
P_A1_25.columns = ["date", "product", "volume", "cement_42_5r", "cement_4av", "arid_04", "arid_8_14", "arid_12_20","water", "recycled_water","ad_complast_mr","ad_dynamon_sx","ad_auramix"]

In [9]:
P_A1_25.columns.tolist()

['date',
 'product',
 'volume',
 'cement_42_5r',
 'cement_4av',
 'arid_04',
 'arid_8_14',
 'arid_12_20',
 'water',
 'recycled_water',
 'ad_complast_mr',
 'ad_dynamon_sx',
 'ad_auramix']

In [10]:
# Frequency of rows where any of two or more of the following products; "cement_42_5r", "cement_3a_42", "cement_4av","cement_4av_1" are !=0 in the same row.
# Frequency of rows where any 2+ of the cement products are !=0 in the same row.
cement_columns = ["cement_42_5r","cement_4av"]

# Count non-zero values per row for these columns.
non_zero_cement_count = (P_A1_25[cement_columns] != 0).sum(axis=1)

# Find rows where 2 or more cement products are non-zero.
condition = non_zero_cement_count >= 2
frequency = condition.sum()

print(f"Frequency of rows where 2 or more cement products are non-zero: {frequency}")
print(f"Percentage: {(frequency / len(P_A1_25) * 100):.2f}%")

Frequency of rows where 2 or more cement products are non-zero: 0
Percentage: 0.00%


In [11]:
# How many rows contain non-zero recycled_water in the dataset?
frequency_recycled_water = (P_A1_25["recycled_water"] != 0).sum()
print(f"Frequency of rows where recycled_water is non-zero: {frequency_recycled_water}")


Frequency of rows where recycled_water is non-zero: 0


In [12]:
# Dropping recycled_water column from the dataset.
P_A1_25 = P_A1_25.drop(columns=["recycled_water"])

In [13]:
## we add a new where all values will be P_A1 because is the future merging we need to identify from which plant the load is come from. 
# Inserting a new column and value. 
# P_A1_24_clean["plant"] = "P_A1"
# Inserting a column position too. 
P_A1_25.insert(1, "plant", "P_A1")

In [14]:
P_A1_25.columns.tolist()

['date',
 'plant',
 'product',
 'volume',
 'cement_42_5r',
 'cement_4av',
 'arid_04',
 'arid_8_14',
 'arid_12_20',
 'water',
 'ad_complast_mr',
 'ad_dynamon_sx',
 'ad_auramix']

---

In [15]:
P_A1_25.to_csv(cfg["P_A1_25_ready_csv_path"], index=False, encoding="utf-8-sig")